In [32]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/raw")
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

customers = pd.read_csv(DATA_DIR / "customers.csv")
transactions = pd.read_csv(DATA_DIR / "transactions.csv")
transaction_items = pd.read_csv(DATA_DIR / "transaction_items.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
support = pd.read_csv(DATA_DIR / "customer_support.csv")
marketing = pd.read_csv(DATA_DIR / "marketing_touchpoints.csv")
stores = pd.read_csv(DATA_DIR / "stores.csv")

In [33]:
def clean_binary_flag(x):
    """Chuẩn hóa cờ nhị phân về {0,1} từ các dạng 0/1, Yes/No, TRUE/FALSE."""
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    if x in ["1", "yes", "true"]:
        return 1
    if x in ["0", "no", "false"]:
        return 0
    return np.nan

def parse_date(series):
    """Parse date với errors='coerce'."""
    return pd.to_datetime(series, errors="coerce")

In [34]:
# --- binary flags ---
marketing["is_opened_clean"] = marketing["is_opened"].apply(clean_binary_flag)
marketing["is_clicked_clean"] = marketing["is_clicked"].apply(clean_binary_flag)

# Enforce is_clicked <= is_opened
marketing.loc[marketing["is_clicked_clean"] == 1, "is_opened_clean"] = 1

# Optional: fill NaN (có thể giữ NaN nếu bạn muốn)
marketing["is_opened_clean"] = marketing["is_opened_clean"].fillna(0)
marketing["is_clicked_clean"] = marketing["is_clicked_clean"].fillna(0)

# --- channel chuẩn hóa ---
valid_mkt_channels = [
    "Email", "SMS", "Push Notification", "Facebook Ad", "Zalo OA", "In-app", "Flash Sale"
]
marketing["channel_clean"] = marketing["channel"].where(
    marketing["channel"].isin(valid_mkt_channels), "Unknown"
)

# --- sent_date ---
marketing["sent_date_clean"] = parse_date(marketing["sent_date"])

# Lưu bản cleaned của marketing
marketing_clean = marketing.copy()
marketing_clean.to_csv(OUTPUT_DIR / "marketing_touchpoints_clean.csv", index=False)

In [35]:
customers["birth_year"] = pd.to_numeric(customers["birth_year"], errors="coerce")

# Giữ năm sinh trong khoảng hợp lý (vd: 1900–2010)
valid_birth_mask = customers["birth_year"].between(1900, 2010)
customers.loc[~valid_birth_mask, "birth_year"] = np.nan

customers["birth_year_missing_flag"] = customers["birth_year"].isna().astype(int)

# Impute bằng median
median_birth_year = customers["birth_year"].median()
customers["birth_year"] = customers["birth_year"].fillna(median_birth_year)

# Ngày đăng ký
if "registration_date" in customers.columns:
    customers["registration_date_clean"] = parse_date(customers["registration_date"])

customers_clean = customers.copy()
customers_clean.to_csv(OUTPUT_DIR / "customers_clean.csv", index=False)

In [36]:
# satisfaction_score
support["satisfaction_score"] = pd.to_numeric(
    support["satisfaction_score"], errors="coerce"
)

# Giữ 1–5, còn lại NaN
valid_score_mask = support["satisfaction_score"].between(1, 5)
support.loc[~valid_score_mask, "satisfaction_score"] = np.nan

# channel
valid_support_channels = [
    "Email", "Chat", "Hotline", "Facebook", "Zalo", "Ti ca hng", "Tại cửa hàng"
]
support["channel_clean"] = support["channel"].where(
    support["channel"].isin(valid_support_channels), "Unknown"
)

# created_date
if "created_date" in support.columns:
    support["created_date_clean"] = parse_date(support["created_date"])

# Flag trường hợp "Pending" nhưng có score (để nêu trong báo cáo)
support["pending_with_score_flag"] = (
    (support["resolution_status"] == "Pending")
    & support["satisfaction_score"].notna()
).astype(int)

support_clean = support.copy()
support_clean.to_csv(OUTPUT_DIR / "customer_support_clean.csv", index=False)

In [37]:
# transactions dates
for col in ["transaction_date"]:
    if col in transactions.columns:
        transactions[f"{col}_clean"] = parse_date(transactions[col])

# transaction_items: ép kiểu số, đặt <= 0 thành NaN
for col in ["quantity", "unit_price"]:
    if col in transaction_items.columns:
        transaction_items[col] = pd.to_numeric(transaction_items[col], errors="coerce")
        transaction_items.loc[transaction_items[col] <= 0, col] = np.nan

# Tạo bản sao và DROP các dòng thiếu quantity hoặc unit_price
transaction_items_clean = transaction_items.copy()
transaction_items_clean = transaction_items_clean.dropna(subset=["quantity", "unit_price"])

# Sao chép transactions sau khi thêm transaction_date_clean
transactions_clean = transactions.copy()

# Ghi ra file
transaction_items_clean.to_csv(OUTPUT_DIR / "transaction_items_clean.csv", index=False)
transactions_clean.to_csv(OUTPUT_DIR / "transactions_clean.csv", index=False)

# products, stores thường ít lỗi; bạn có thể chỉ parse date, chuẩn hóa text nếu cần
stores["opened_date_clean"] = parse_date(stores["opened_date"])
stores_clean = stores.copy()
stores_clean.to_csv(OUTPUT_DIR / "stores_clean.csv", index=False)

products_clean = products.copy()
products_clean.to_csv(OUTPUT_DIR / "products_clean.csv", index=False)

In [38]:
# ════════════════════════════════════════════════════════════════
# MISSING VALUES — Chiến lược rõ ràng theo từng bảng
# ════════════════════════════════════════════════════════════════

# ── customers ──────────────────────────────────────────────────
# gender: categorical → impute bằng mode (không drop vì chỉ 5.4%)
customers["gender"] = customers["gender"].fillna(customers["gender"].mode()[0])

# city, acquisition_channel: categorical → giữ NaN thành 'Unknown'
# (không impute vì không có cơ sở dự đoán location)
customers["city"] = customers["city"].fillna("Unknown")
customers["acquisition_channel"] = customers["acquisition_channel"].fillna("Unknown")

# registration_date: chỉ 2% thiếu → drop các dòng này
# (ngày đăng ký là dữ liệu gốc, không impute được)
customers = customers.dropna(subset=["registration_date"])
print(f"customers sau drop missing registration_date: {len(customers)} rows")

# ── transactions ───────────────────────────────────────────────
# payment_method: categorical 5.4% → impute mode
transactions["payment_method"] = transactions["payment_method"].fillna(
    transactions["payment_method"].mode()[0]
)

# discount_applied: 4.7% → impute bằng 0 (giả định không có giảm giá)
transactions["discount_applied"] = transactions["discount_applied"].fillna(0)

# ── transaction_items ──────────────────────────────────────────
# quantity, unit_price: ~5% → drop dòng (không thể tính revenue nếu thiếu)
items_before = len(transaction_items)
transaction_items = transaction_items.dropna(subset=["quantity", "unit_price"])
print(f"transaction_items: dropped {items_before - len(transaction_items)} rows missing qty/price")

# ── marketing_touchpoints ──────────────────────────────────────
# offer_value: 5.2% → impute bằng 0 (không có offer = offer = 0)
marketing["offer_value"] = marketing["offer_value"].fillna(0)

# channel: 5.6% → đã map Unknown ở trên, giữ nguyên
# is_opened, is_clicked: đã xử lý ở clean_binary_flag fillna(0)

# ── customer_support ───────────────────────────────────────────
# satisfaction_score: 28.9% thiếu → KHÔNG impute
# (vì missing có nghĩa: ticket chưa đánh giá, impute sẽ sai lệch phân tích)
# Giữ NaN — chỉ dùng khi tính avg_satisfaction với skipna=True

# issue_type: 6% → impute "Unknown"
support["issue_type"] = support["issue_type"].fillna("Unknown")

# channel: 4.1% → impute "Unknown"
support["channel"] = support["channel"].fillna("Unknown")

# ── products ───────────────────────────────────────────────────
# brand: 6% thiếu → impute "Unknown Brand"
products["brand"] = products["brand"].fillna("Unknown Brand")

print("✅ Missing values handled")

customers sau drop missing registration_date: 784 rows
transaction_items: dropped 794 rows missing qty/price
✅ Missing values handled


In [39]:
# ════════════════════════════════════════════════════════════════
# GIÁ TRỊ KHÔNG HỢP LỆ
# ════════════════════════════════════════════════════════════════

# ── gender: 'X', '0', 'Unknown' → map về 'Other' ──────────────
valid_genders = ['M', 'F', 'Other']
customers["gender_clean"] = customers["gender"].where(
    customers["gender"].isin(valid_genders), "Other"
)
print("gender value counts sau clean:")
print(customers["gender_clean"].value_counts())

# ── payment_method: 'Bitcoin', '???', 'Unknown' → 'Other' ──────
valid_payments = ['Cash', 'Credit Card', 'Bank Transfer', 'VNPay', 'ZaloPay', 'Momo']
transactions["payment_method_clean"] = transactions["payment_method"].where(
    transactions["payment_method"].isin(valid_payments), "Other"
)
print("\npayment_method value counts sau clean:")
print(transactions["payment_method_clean"].value_counts())

gender value counts sau clean:
gender_clean
M        411
F        355
Other     18
Name: count, dtype: int64

payment_method value counts sau clean:
payment_method_clean
Momo             789
Credit Card      597
ZaloPay          595
VNPay            576
Bank Transfer    571
Cash             270
Other            102
Name: count, dtype: int64


In [40]:
# ════════════════════════════════════════════════════════════════
# LOGIC CONSISTENCY: Pending ticket không được có satisfaction
# ════════════════════════════════════════════════════════════════

# Đếm vi phạm
pending_with_score = (
    (support["resolution_status"] == "Pending") &
    support["satisfaction_score"].notna()
).sum()
print(f"Pending tickets có satisfaction_score: {pending_with_score}")

# Fix: set về NaN (logic: Pending = chưa giải quyết = không có đánh giá)
support.loc[support["resolution_status"] == "Pending", "satisfaction_score"] = np.nan
print("✅ Đã xóa satisfaction_score của Pending tickets")

# Kiểm tra lại
assert (
    (support["resolution_status"] == "Pending") &
    support["satisfaction_score"].notna()
).sum() == 0, "Vẫn còn Pending có score!"
print("✅ Logic check passed")

Pending tickets có satisfaction_score: 0
✅ Đã xóa satisfaction_score của Pending tickets
✅ Logic check passed


### Cleaning summary

- `marketing.is_opened` and `marketing.is_clicked` contained multiple encodings
  (`Yes`, `TRUE`, `2`, `-1`). We standardized them to binary flags {0,1} using a
  unified mapping function and enforced the business rule `is_clicked <= is_opened`.
- `customers.birth_year` had impossible values (0, negatives, very large years).
  We kept only years between 1900 and 2010, set invalid values to missing, then
  imputed with the median and added a missingness flag.
- `customer_support.satisfaction_score` was designed to be on a 1–5 scale but
  included values such as 0, 10, 99 and -1. We treated these as invalid and set
  them to missing. Pending tickets were allowed to have missing scores.
- All date fields were parsed with `errors='coerce'`, converting malformed dates
  like `0000-00-00` or `NaT`-like strings into missing values.
- Categorical channels in both marketing and support were normalized to a
  whitelist of valid channels, with others mapped to `Unknown`.